# SRA Runs Exploration

In [1]:
import pandas as pd
from pysradb import SRAweb

# Initialize the SRA metadata web interface
sra = SRAweb()

# Fetch all metadata for the project (e.g., SRP accession or PRJNA accession)
project_id = "SRP559264"  # <--- Replace with your study accession
df_metadata = sra.sra_metadata(project_id)

# Display available columns to see where sample attributes are stored
print(f"Total runs found: {len(df_metadata)}")
print("Columns:", df_metadata.columns.tolist())

/home/w/anaconda3/envs/eva/lib/python3.11/site-packages/pysradb/utils.py:14: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Total runs found: 32
Columns: ['study_accession', 'study_title', 'experiment_accession', 'experiment_title', 'experiment_desc', 'organism_taxid', 'organism_name', 'library_name', 'library_strategy', 'library_source', 'library_selection', 'library_layout', 'sample_accession', 'sample_title', 'biosample', 'bioproject', 'instrument', 'instrument_model', 'instrument_model_desc', 'total_spots', 'total_size', 'run_accession', 'run_total_spots', 'run_total_bases']


In [3]:
# Select relevant columns for sample identification
cols_to_view = ['run_accession', 'sample_accession', 'experiment_title', 'study_title']
# Filter columns that actually exist in the fetched DataFrame
available_cols = [c for c in cols_to_view if c in df_metadata.columns]

df_metadata[available_cols]

,run_accession,sample_accession,experiment_title,study_title
0,SRR32086830,SRS23862427,"GSM8750892: SC, 17DMD, differentiation, biol r...",Multi-modal analysis of satellite cells reveal...
1,SRR32086831,SRS23862427,"GSM8750892: SC, 17DMD, differentiation, biol r...",Multi-modal analysis of satellite cells reveal...
2,SRR32086832,SRS23862426,"GSM8750891: SC, 16DMD, differentiation, bio re...",Multi-modal analysis of satellite cells reveal...
3,SRR32086833,SRS23862426,"GSM8750891: SC, 16DMD, differentiation, bio re...",Multi-modal analysis of satellite cells reveal...
4,SRR32086834,SRS23862422,"GSM8750890: SC, 15DMD, differentation, biol re...",Multi-modal analysis of satellite cells reveal...
5,SRR32086835,SRS23862422,"GSM8750890: SC, 15DMD, differentation, biol re...",Multi-modal analysis of satellite cells reveal...
6,SRR32086836,SRS23862424,"GSM8750889: SC, 14DMD, differentiation, biol r...",Multi-modal analysis of satellite cells reveal...
7,SRR32086837,SRS23862424,"GSM8750889: SC, 14DMD, differentiation, biol r...",Multi-modal analysis of satellite cells reveal...
8,SRR32086838,SRS23862421,"GSM8750888: SC, 17DMD, proliferation, biol rep...",Multi-modal analysis of satellite cells reveal...
9,SRR32086839,SRS23862421,"GSM8750888: SC, 17DMD, proliferation, biol rep...",Multi-modal analysis of satellite cells reveal...


In [5]:
# Filter for Differentiation state only
diff_df = df_metadata[df_metadata['experiment_title'].str.contains('differentiation', case=False, na=False)].copy()

# Deduplicate to pick 1 technical run per biological sample (GSM)
diff_unique = diff_df.drop_duplicates(subset=['sample_accession'])

# Select 3 WT and 3 DMD biological samples
wt_diff_runs = diff_unique[diff_unique['experiment_title'].str.contains('WT', case=False)]['run_accession'].head(3).tolist()
dmd_diff_runs = diff_unique[diff_unique['experiment_title'].str.contains('DMD', case=False)]['run_accession'].head(3).tolist()

selected_6_runs = wt_diff_runs + dmd_diff_runs

print("Selected 3 WT Differentiation Runs: ", wt_diff_runs)
print("Selected 3 DMD Differentiation Runs:", dmd_diff_runs)
print("Final 6-Run SRA List:", selected_6_runs)

Selected 3 WT Differentiation Runs:  ['SRR32086846', 'SRR32086848', 'SRR32086850']
Selected 3 DMD Differentiation Runs: ['SRR32086830', 'SRR32086832', 'SRR32086836']
Final 6-Run SRA List: ['SRR32086846', 'SRR32086848', 'SRR32086850', 'SRR32086830', 'SRR32086832', 'SRR32086836']


# Exploring the Downloaded fastq files 

In [4]:
import glob
import os
import pandas as pd

# Target directory relative to /home/w/EVA/notebooks
data_dir = "../data/raw/fastq/"

runs = {
    "SRR32086846": "WT",
    "SRR32086848": "WT",
    "SRR32086850": "WT",
    "SRR32086830": "DMD",
    "SRR32086832": "DMD",
    "SRR32086836": "DMD",
}


def inspect_fastq(filepath):
    read_count = 0
    read_length = 0
    with open(filepath, "r") as handle:
        for i, line in enumerate(handle):
            if i % 4 == 1:  # Sequence line
                if read_count == 0:
                    read_length = len(line.strip())
                read_count += 1
    return read_count, read_length


file_stats = []
for run_id, condition in runs.items():
    matched_files = glob.glob(
        os.path.join(data_dir, f"**/*{run_id}*.fastq*"), recursive=True
    )

    for filepath in matched_files:
        size_gb = os.path.getsize(filepath) / (1024**3)
        reads, length = inspect_fastq(filepath)
        file_stats.append(
            {
                "Run": run_id,
                "Condition": condition,
                "File Name": os.path.basename(filepath),
                "Size (GB)": round(size_gb, 2),
                "Total Reads (M)": round(reads / 1e6, 2),
                "Read Length (bp)": length,
            }
        )

summary_df = pd.DataFrame(file_stats)
summary_df

,Run,Condition,File Name,Size (GB),Total Reads (M),Read Length (bp)
0,SRR32086846,WT,SRR32086846_3.fastq,1.79,6.72,74
1,SRR32086848,WT,SRR32086848_3.fastq,1.84,6.84,74
2,SRR32086850,WT,SRR32086850_3.fastq,1.67,6.32,74
3,SRR32086830,DMD,SRR32086830_3.fastq,1.16,4.32,74
4,SRR32086832,DMD,SRR32086832_3.fastq,1.59,5.95,74
5,SRR32086836,DMD,SRR32086836_3.fastq,1.77,6.72,74


In [6]:
import gzip
import os

# Define paths for the two files you want to inspect
target_files = [
    "../data/raw/fastq/SRR32086832_3.fastq",
    "../data/raw/fastq_cleaned/SRR32086832.clean.fastq",  # Replace with your second file path
]

for target_file in target_files:
    # Get just the file name for a clean header
    filename = os.path.basename(target_file)

    # Print header message
    print(f"\n{'='*10} First 5 lines of: {filename} {'='*10}")

    # Determine whether to use gzip based on file extension
    open_fn = gzip.open if target_file.endswith(".gz") else open

    # Read and print the first 5 lines
    with open_fn(target_file, "rt") as handle:
        for _ in range(4):
            line = handle.readline()
            if not line:
                break
            print(line.strip())


========== First 5 lines of: SRR32086832_3.fastq ==========
@SRR32086832.1 NDX550280:176:H3GNNBGXT:1:11101:9201:1052 length=74
GNNNNGNCNTNGCNNCNCNNNNNNNNNNNNNNNNNNNNNNNNNNNNNANNNNNNANNNNNNNNNNNNNNNNNNN
+SRR32086832.1 NDX550280:176:H3GNNBGXT:1:11101:9201:1052 length=74
A####E#E#E#EE##E#E#############################<######A###################

========== First 5 lines of: SRR32086832.clean.fastq ==========
@SRR32086832.9 NDX550280:176:H3GNNBGXT:1:11101:8512:1083 length=75
GTCCTCTGCTGGGGCCGGGGTGCTGCTCCCTCCCACGGTGCCAATGCCTGTGTGTGTTGTGTCTGTGAGAAGTCC
+SRR32086832.9 NDX550280:176:H3GNNBGXT:1:11101:8512:1083 length=75
AAAAAEEEEEEEEEEEEEAEAEEEEAEEEAEEEE<EA6EAEE/</EEA6EA/66E6AA666EAA/A/A/EE/AE6


# 